### Notebook to use the ArgoFormatChecker provided by AMRIT Consortium.

The Argo NetCDF file format checker performs format and content checks on Argo NetCDF files.
The Argo NetCDF format is described in "Argo user's manual" http://dx.doi.org/10.13155/29825
More information on https://www.argodatamgt.org/Documentation

In [ ]:
#!pip install ipywidgets

import requests
import json
import os
from pathlib import Path
import pandas as pd
import time
from ipywidgets import FileUpload, VBox, Output
from IPython.display import display, HTML


print("Argo File Checker API Testing")
print("=" * 50)

# ===== CONFIGURATION =====

# Update these settings for your environment. This is BODC test instance configuration.
# Test configuration
API_BASE_URL= "https://livbodcinttst.bodc.me/livkrakentst-1/ewetchy/amrit/argo-toolbox/api/file-checker"
DEFAULT_DAC = "bodc" 

# API endpoints
CHECK_FILE_ENDPOINT = "/check-files"
HEALTH_ENDPOINT = "/"
    
TIMEOUT = 30  # API request timeout in seconds
HEADERS = {
    'accept': 'application/json'
}

print(f"API Base URL: {API_BASE_URL}")
print(f"DAC: {DEFAULT_DAC}")

Argo File Checker API Testing
API Base URL: https://livbodcinttst.bodc.me/livkrakentst-1/ewetchy/amrit/argo-toolbox/api/file-checker
DAC: bodc


In [2]:
def test_api_connection():
    """Test if the API is accessible."""
    
    print("\nTesting API Connection..")
    print("-" * 30)
    
    try:
        response = requests.get(f"{API_BASE_URL}/", timeout=5)
        if response.status_code == 200:
            result = response.json()
            print(f"API is accessible!")
            print(f"Health Check Response: {result}")
            return True
        else:
            print(f"API returned status code: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except requests.exceptions.ConnectionError as error:
        print("Could not connect to API. Is the container running?")
        print(error)
        return False
    except requests.exceptions.Timeout:
        print("Connection timed out")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

In [3]:
def file_check(file_paths, dac=DEFAULT_DAC, verbose=True):
    """Checking files using the filechecker."""
    
   # Prepare files for upload
    files_data = []
    FILE_CHECK_URL = f"{API_BASE_URL}/{CHECK_FILE_ENDPOINT}"
    
    for file_path in file_paths:
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
            
        
        # Determine content type based on file extension
        if file_path.endswith('.json'):
            content_type = 'application/json'
        elif file_path.endswith('.nc'):
            content_type = 'application/octet-stream'
        else:
            content_type = 'application/octet-stream'

        try:
            files_data.append((
                'files',
                (os.path.basename(file_path), open(file_path, 'rb'), content_type)
            ))
        except Exception as e:
            print(f"Error opening {file_path}: {e}")
            continue
    
    if not files_data:
        print("No valid files to upload!")
        return None

    params = {'dac': dac}
    
    try:
        start_time = time.time()
        
        if verbose:
            print(f"Sending POST request...")
        
        response = requests.post(
            FILE_CHECK_URL,
            files=files_data,
            params=params,
            headers=HEADERS,
            timeout=TIMEOUT
        )
        
        end_time = time.time()
        processing_time = end_time - start_time
        
        # Close file handles
        for _, (_, file_handle, _) in files_data:
            file_handle.close()

        if response.status_code == 200:
            try:
                result = response.json()
                if verbose:
                    print("Request successful!")
                return {
                    'success': True,
                    'status_code': response.status_code,
                    'processing_time': processing_time,
                    'result': result
                }
            except json.JSONDecodeError:
                if verbose:
                    print("Response is not valid JSON")
                return {
                    'success': False,
                    'status_code': response.status_code,
                    'processing_time': processing_time,
                    'result': response.text
                }
        else:
            if verbose:
                print(f"Request failed with status {response.status_code}")
                print(f"Response: {response.text[:500]}...")
            
            return {
                'success': False,
                'status_code': response.status_code,
                'processing_time': processing_time,
                'result': response.text
            }
            
    except requests.exceptions.Timeout:
        if verbose:
            print(f"Request timed out after {TIMEOUT} seconds")
        return {'success': False, 'error': 'timeout'}
        
    except requests.exceptions.RequestException as e:
        if verbose:
            print(f"Request error: {e}")
        return {'success': False, 'error': str(e)}
        
    except Exception as e:
        if verbose:
            print(f"Unexpected error: {e}")
        return {'success': False, 'error': str(e)}
        

In [4]:
def show_result(result: dict):
    """Display the result dict in a neat dataframe."""
    if not isinstance(result, dict):
        print(result)
        return
    if not result.get("success"):
        print("Request failed")
        print(f"Status: {result.get('status_code')}, Time: {result.get('processing_time')}")
        print(result.get("result"))
        return
        
    results = result.get("result", {}).get("results", [])
    if not results:
        print("No results found.")
        return
        
    for r in results:
        r["errors_messages"] = "\n".join(r.get("errors_messages", []))
        r["warnings_messages"] = "\n".join(r.get("warnings_messages", []))
    df = pd.DataFrame(results, columns=[
        "file",
        "result",
        "phase",
        "errors_number",
        "warnings_number",
        "errors_messages",
        "warnings_messages"
    ])
    
   # CSS style for borders and wrapping
    styles = """
    <style>
        table {
            border: 1px solid black;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black !important;
            padding: 5px;
            text-align: left;
            vertical-align: top;
            max-width: 400px;
            white-space: pre-wrap;
            word-wrap: break-word;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
    """
    
    html_table = df.to_html(escape=False).replace("\\n", "<br>")
    display(HTML(styles + html_table))


In [5]:
# Test 1: Check API connection
api_available = test_api_connection()

if not api_available:
    print("\nAPI is not accessible. Please check:")
else:
    print("\nAPI is ready for testing!")


Testing API Connection..
------------------------------
API is accessible!
Health Check Response: ['OK']

API is ready for testing!


In [ ]:
# Interactive file upload widget - if working with Datascience platform

upload = FileUpload(accept='.json,.nc', multiple=True)
out = Output()

def on_upload_change(change):
    file_paths = []
    for fileinfo in upload.value:
        fname = fileinfo["name"]
        tmp_path = f"/tmp/{fname}"
        with open(tmp_path, "wb") as f:
            f.write(fileinfo["content"])
        file_paths.append(tmp_path)

    with out:
        out.clear_output()
        print(f"Selected files: {file_paths}")
        res=file_check(file_paths)
        show_result(res)

upload.observe(on_upload_change, names="value")

display(VBox([upload, out]))

In [ ]:
# No Interactive file upload widget - if working with VS code notebook

#file_paths = ["C:/Users/vidkri/Documents/ARGO/RBR/D5906457_077.nc"]
file_paths = ["/path/to/ncfiles.nc"]
    
if file_paths:
    res = file_check(file_paths)
    show_result(res)


Sending POST request...
Request successful!


,file,result,phase,errors_number,warnings_number,errors_messages,warnings_messages
0,/home/app/input/c036d0b2-5d23-46c4-b831-e23a882d430f/D5906457_077.nc,FILE-REJECTED,FILE-NAME-CHECK,3,0,DATA_CENTRE[1]: 'AO': Invalid for DAC 'BODC' DATA_CENTRE[2]: 'AO': Invalid for DAC 'BODC' DATA_CENTRE[3]: 'AO': Invalid for DAC 'BODC',
